# dw 教学数仓探查

本地 `docker compose -f docker/docker-compose.yaml` 起的 MySQL 8.0（容器 `mysql`，宿主机端口 `3306`）：
`dw` 是教学数仓，`meta` 是元数据库。账号默认 `early / dili123`（见 `conf/app_config.yaml` 的 `db_dw` 段）。

- 数据覆盖 **2024-01-01 ~ 2025-12-31**，所以“今天”固定取 `2025-12-31`
- 每个 cell 都能单独运行；改完 SQL 按 `Shift+Enter` 重跑

## 0. 连接

In [1]:
import os

import pandas as pd
from sqlalchemy import create_engine

# 默认值对齐 conf/app_config.yaml 的 db_dw；要连别处就设环境变量
DW = {
    "host": os.getenv("DW_HOST", "127.0.0.1"),
    "port": int(os.getenv("DW_PORT", "3306")),
    "user": os.getenv("DW_USER", "early"),
    "password": os.getenv("DW_PASSWORD", "dili123"),
    "database": os.getenv("DW_DATABASE", "dw"),
}
engine = create_engine(
    "mysql+mysqlconnector://{user}:{password}@{host}:{port}/{database}".format(**DW),
    pool_pre_ping=True,
)


def q(sql: str, **kwargs) -> pd.DataFrame:
    # 执行一条 SQL 并返回 DataFrame；kwargs 作为查询参数传给驱动
    with engine.connect() as conn:
        return pd.read_sql(sql, conn, params=kwargs or None)

In [2]:
from sqlalchemy import inspect
insp = inspect(engine)

def get_table_columns(table_name:str) -> str:
    columns = insp.get_columns(table_name)
    col_names = [col["name"] for col in columns]
    return ",".join(col_names)

In [ ]:
get_table_columns("dim_shop")


'shop_id,shop_name,shop_type,main_category_l1,shop_score,open_date'

In [ ]:
dim_shop = insp.get_columns("dim_shop")
dim_shop

[{'name': 'shop_id',
  'type': VARCHAR(collation='utf8mb4_general_ci', length=20),
  'default': None,
  'comment': '店铺ID',
  'nullable': False},
 {'name': 'shop_name',
  'type': VARCHAR(collation='utf8mb4_general_ci', length=50),
  'default': None,
  'comment': '店铺名称',
  'nullable': False},
 {'name': 'shop_type',
  'type': VARCHAR(collation='utf8mb4_general_ci', length=10),
  'default': None,
  'comment': '店铺类型：自营/旗舰店/专营店/专卖店',
  'nullable': False},
 {'name': 'main_category_l1',
  'type': VARCHAR(collation='utf8mb4_general_ci', length=20),
  'default': None,
  'comment': '主营一级类目',
  'nullable': False},
 {'name': 'shop_score',
  'type': DECIMAL(precision=3, scale=2),
  'default': None,
  'comment': '店铺综合评分',
  'nullable': False},
 {'name': 'open_date',
  'type': DATE(),
  'default': None,
  'comment': '开店日期',
  'nullable': False}]

## 1. 表清单 + 注释

In [ ]:
q("""
SELECT table_name, table_rows, table_comment
FROM information_schema.tables
WHERE table_schema = 'dw'
ORDER BY table_name
""")

,TABLE_NAME,TABLE_ROWS,TABLE_COMMENT
0,dim_activity,14,营销活动维度
1,dim_category,40,类目维度（三级类目）
2,dim_date,731,日期维度
3,dim_region,44,地区维度（城市级）
4,dim_shop,49,店铺维度
5,dim_sku,317,商品维度（SKU）
6,dim_user,3000,用户维度
7,fact_order,58853,订单事实表（订单粒度）
8,fact_order_item,85450,订单明细事实表（订单行粒度）
9,fact_refund,5780,退款事实表（退款申请粒度）


In [ ]:
q("""
select full_date from dim_date order by date_id limit 20
""")

,full_date
0,2024-01-01
1,2024-01-02
2,2024-01-03
3,2024-01-04
4,2024-01-05
5,2024-01-06
6,2024-01-07
7,2024-01-08
8,2024-01-09
9,2024-01-10


In [ ]:
get_table_columns("dim_date")

'date_id,full_date,year,quarter,month,day,week_of_year,day_of_week,is_weekend,is_holiday,holiday_name'

## 2. 指标口径速算

`fact_order` 是订单头粒度（一行一个订单），直接聚合不会扇出。

> 坑：订单头金额一旦 join 到 `fact_order_item`（一行一个订单行）再求和，就会被放大。

In [ ]:
q("""
SELECT
  ROUND(SUM(order_amount), 2)                                          AS 下单金额,
  ROUND(SUM(CASE WHEN pay_time IS NOT NULL THEN order_amount END), 2)   AS 支付金额,
  COUNT(DISTINCT CASE WHEN pay_time IS NOT NULL THEN user_id END)        AS 支付买家数,
  ROUND(SUM(CASE WHEN pay_time IS NOT NULL THEN order_amount END)
        / NULLIF(COUNT(DISTINCT CASE WHEN pay_time IS NOT NULL THEN user_id END), 0), 2) AS 客单价
FROM fact_order
""")

,下单金额,支付金额,支付买家数,客单价
0,90731816.04,83571534.04,2958,28252.72


In [ ]:
q("""
select 
t1.shop_id,
sum(t1.order_amount) as gmv
from fact_order t1 
left join
dim_shop t2
on t1.shop_id = t2.shop_id
group by t1.shop_id
""")

In [ ]:
q("""
select 
timestampdiff(day,date("2024-01-01 06:08:13"),date("2024-01-02 07:19:44")) as nday
""")

,nday
0,1


## 3. 月度趋势（2024 vs 2025 同月）

销售额按**支付时间**切时间，所以用 `pay_time` 而不是 `order_time`：
下单与支付跨月/跨年的订单，按 `order_time` 分组会被算到下单月去。

In [ ]:
q("""
select 
month(pay_time) as month,
round(sum(case when year(pay_time) = "2024" then order_amount end),2) as y2024,
round(sum(case when year(pay_time) = "2025" then order_amount end),2) as y2025
from 
fact_order 
where pay_time is not null 
group by month(pay_time)
order by month asc
""")

,month,y2024,y2025
0,1,2719422.29,3537898.29
1,2,1762452.69,2347963.84
2,3,2639090.51,3646273.82
3,4,2078957.52,3142856.31
4,5,2615455.46,3441754.21
5,6,4624459.57,5611820.03
6,7,2297732.49,2749276.24
7,8,2650814.03,3069100.46
8,9,2876374.97,3296570.33
9,10,4050998.12,4858566.63


#### `fact_order` 订单事实表（订单粒度）

| 字段 | 类型 | 说明 |
| --- | --- | --- |
| `order_id` | VARCHAR(30), PK | 订单号 |
| `user_id` | FK → `dim_user` | 下单用户 |
| `shop_id` | FK → `dim_shop` | 店铺 |
| `receiver_region_id` | FK → `dim_region` | **收货城市**（同一张 `dim_region` 被用户常住地和订单收货地两种角色使用） |
| `order_date_id` | FK → `dim_date` | 下单日期 |
| `order_time` | DATETIME | 下单时间 |
| `pay_date_id` | FK → `dim_date`，可空 | 支付日期，未支付为 NULL（**同一张 `dim_date` 有多种日期角色**） |
| `pay_time` | DATETIME，可空 | 支付时间 |
| `platform` | VARCHAR(10) | 下单终端：APP / 小程序 / PC / H5 |
| `payment_method` | VARCHAR(10)，可空 | 支付方式：支付宝 / 微信支付 / 银行卡 / 花呗分期 |
| `order_status` | VARCHAR(10) | 订单状态：待支付 / 已取消 / 已支付 / 已发货 / 已完成（退款单独记在 `fact_refund`，不改订单状态） |
| `item_quantity` | INT | 商品总件数 |
| `total_amount` | DECIMAL | 商品总额 = Σ 明细 `item_amount` |
| `discount_amount` | DECIMAL | 优惠总额 = Σ 明细 `discount_amount` |
| `freight_amount` | DECIMAL | 运费 |
| `order_amount` | DECIMAL | 订单应付金额 = `total_amount − discount_amount + freight_amount`；只有 `pay_time` 不为空时才是实际支付的金额 |

#### `dim_sku` 商品维度

| 字段 | 类型 | 说明 |
| --- | --- | --- |
| `sku_id` | VARCHAR(20), PK | SKU ID |
| `spu_id` | VARCHAR(20) | SPU ID（同一款商品的不同规格共享一个 SPU） |
| `sku_name` | VARCHAR(100) | 商品名称（含规格），如“iPhone 16 Pro 256GB 黑色钛金属” |
| `category_id` | VARCHAR(20), FK → `dim_category` | 三级类目 |
| `brand_name` | VARCHAR(30) | 品牌 |
| `shop_id` | VARCHAR(20), FK → `dim_shop` | 所属店铺 |
| `list_price` | DECIMAL(12,2) | 标价（划线价） |
| `cost_price` | DECIMAL(12,2) | 成本价，用于计算毛利 |
| `launch_date` | DATE | 上架日期 |
| `sku_status` | VARCHAR(10) | 状态：在售 / 下架 |

#### `fact_order_item` 订单明细事实表（订单行粒度）

| 字段 | 类型 | 说明 |
| --- | --- | --- |
| `order_item_id` | VARCHAR(30), PK | 订单明细 ID |
| `order_id` | FK → `fact_order` | 所属订单（下单时间、状态、用户都要通过订单表获取） |
| `sku_id` | FK → `dim_sku` | 商品 |
| `activity_id` | FK → `dim_activity`，可空 | 参与的活动，未参与为 NULL |
| `quantity` | INT | 购买件数 |
| `unit_price` | DECIMAL | 成交单价（可能低于标价） |
| `item_amount` | DECIMAL | 商品金额 = `unit_price × quantity` |
| `discount_amount` | DECIMAL | 分摊到本行的优惠 |
| `pay_amount` | DECIMAL | 本行实付金额 = `item_amount − discount_amount`，不含运费 |

In [ ]:
query = """
with sku_sales as (
select c.category_l1,
s.sku_id,
sum(i.pay_amount) as sales
from fact_order_item i 
join fact_order o on o
)
"""

'order_id,user_id,shop_id,receiver_region_id,order_date_id,order_time,pay_date_id,pay_time,platform,payment_method,order_status,item_quantity,total_amount,discount_amount,freight_amount,order_amount'

In [ ]:
q(query)

In [ ]:
q("""
select order_id,order_time,user_id
from 
fact_order
where pay_time is null limit 1
""")

,order_id,order_time,user_id
0,O2024010100001,2024-01-01 04:48:42,U002311


In [ ]:
q("""
select * from fact_order_item where order_id ="O2024010100003"
""")

,order_item_id,order_id,sku_id,activity_id,quantity,unit_price,item_amount,discount_amount,pay_amount
0,O2024010100003-1,O2024010100003,K00089,A13,1,609.0,609.0,5.21,603.79
1,O2024010100003-2,O2024010100003,K00093,A13,2,279.0,558.0,4.79,553.21


## 4. 类目 × 用户等级交叉

类目走雪花：`fact_order_item → dim_sku → dim_category`；金额用明细的 `pay_amount`（不含运费）。

In [ ]:
get_table_columns("fact_order_item")

'order_item_id,order_id,sku_id,activity_id,quantity,unit_price,item_amount,discount_amount,pay_amount'

In [ ]:
q("""
select order_id,sku_id,cnt
from 
(select order_id,sku_id,count(order_id) over(partition by order_id) cnt
from fact_order_item 
group by order_id,sku_id) t
where cnt >= 3
""")

,order_id,sku_id,cnt
0,O2024010100007,K00220,3
1,O2024010100007,K00221,3
2,O2024010100007,K00223,3
3,O2024010100021,K00162,3
4,O2024010100021,K00163,3
...,...,...,...
12565,O2025123100066,K00158,3
12566,O2025123100066,K00160,3
12567,O2025123100069,K00111,3
12568,O2025123100069,K00115,3


In [ ]:
q("""
SELECT c.category_l1, u.member_level,
       ROUND(SUM(i.pay_amount), 2) AS sales
FROM fact_order_item i
JOIN fact_order    o ON o.order_id = i.order_id AND o.pay_time IS NOT NULL
JOIN dim_sku       s ON s.sku_id = i.sku_id
JOIN dim_category  c ON c.category_id = s.category_id
JOIN dim_user      u ON u.user_id = o.user_id
GROUP BY c.category_l1, u.member_level
ORDER BY c.category_l1, sales DESC
""")

,category_l1,member_level,sales
0,家居日用,铂金,694367.10
1,家居日用,金卡,689109.23
2,家居日用,普通会员,638376.11
3,家居日用,银卡,530082.78
4,家居日用,钻石,525855.93
5,家用电器,铂金,4423311.71
6,家用电器,金卡,4252912.89
7,家用电器,普通会员,4073587.68
8,家用电器,银卡,3597784.63
9,家用电器,钻石,3462663.26


## 5. 行为 → 支付漏斗

`dim_sku` **没有** `category_l1`（类目在 `dim_category`）。并且要先把两侧各自聚合到“类目”再 join：
行为表和订单表直接 join 会互相放大，`COUNT(DISTINCT o.user_id)` 还会把“在该类目下支付”算成“付过任意商品”。

> `fact_user_behavior` 只有 浏览 / 收藏 / 加购，**没有“下单”**；支付要回到订单表。

In [ ]:
q("""
WITH beh AS (
  SELECT c.category_l1,
         COUNT(DISTINCT CASE WHEN b.behavior_type = '浏览' THEN b.user_id END) AS browse_users,
         COUNT(DISTINCT CASE WHEN b.behavior_type = '加购' THEN b.user_id END) AS cart_users
  FROM fact_user_behavior b
  JOIN dim_sku      s ON s.sku_id = b.sku_id
  JOIN dim_category c ON c.category_id = s.category_id
  GROUP BY c.category_l1
), pay AS (
  SELECT c.category_l1, COUNT(DISTINCT o.user_id) AS pay_users
  FROM fact_order_item i
  JOIN fact_order   o ON o.order_id = i.order_id AND o.pay_time IS NOT NULL
  JOIN dim_sku      s ON s.sku_id = i.sku_id
  JOIN dim_category c ON c.category_id = s.category_id
  GROUP BY c.category_l1
)
SELECT b.category_l1,
       b.browse_users AS 浏览用户,
       b.cart_users   AS 加购用户,
       COALESCE(p.pay_users, 0) AS 支付用户,
       ROUND(COALESCE(p.pay_users, 0) / NULLIF(b.browse_users, 0) * 100, 2) AS 浏览到支付pct
FROM beh b
LEFT JOIN pay p ON p.category_l1 = b.category_l1
ORDER BY b.category_l1
""")

In [16]:
q("""
select order_status,is_pay 
from 
(select order_id,order_status,case when pay_time is null then 0 else 1 end as is_pay
from 
fact_order ) t
group by 1,2
""")

,order_status,is_pay
0,已取消,0
1,已完成,1
2,已发货,1
3,已支付,1
4,待支付,0


,order_id,rn


In [28]:
q("""
with t1 as (select i.order_item_id,i.order_id,i.quantity,i.pay_amount,r.refund_status,r.refund_quantity,r.refund_amount
from 
fact_order_item i
join fact_refund r on i.order_item_id = r.order_item_id ),

t2 as (select 
o.order_id,t1.order_item_id,t1.refund_amount,o.pay_time
from 
fact_order o 
join t1
on o.order_id = t1.order_id)

select *
from 
(select *,row_number() over(partition by order_id order by order_item_id asc) rn
from t2) tt
where order_id = "O2024010700019"
""")

,order_id,order_item_id,refund_amount,pay_time,rn
0,O2024010700019,O2024010700019-1,1169.69,2024-01-07 11:30:13,1
1,O2024010700019,O2024010700019-2,237.31,2024-01-07 11:30:13,2


## 6. 退款率（按类目）

分母必须是**该类目全部已支付明细**的金额。原写法从 `fact_refund` 出发 join 明细，
分母只剩“有退款的明细”，算出来会接近 90%。

退款只认 `refund_status = '退款成功'`（`退款中 / 已拒绝 / 已撤销` 都不算）。

In [ ]:
q("""
SELECT c.category_l1,
       ROUND(SUM(COALESCE(r.refund_amount, 0)) / NULLIF(SUM(i.pay_amount), 0) * 100, 2) AS 退款率pct
FROM fact_order_item i
JOIN fact_order   o ON o.order_id = i.order_id AND o.pay_time IS NOT NULL
JOIN dim_sku      s ON s.sku_id = i.sku_id
JOIN dim_category c ON c.category_id = s.category_id
LEFT JOIN fact_refund r
       ON r.order_item_id = i.order_item_id AND r.refund_status = '退款成功'
GROUP BY c.category_l1
ORDER BY 退款率pct DESC
""")

,category_l1,退款率pct
0,服饰鞋包,11.54
1,美妆个护,4.42
2,家居日用,4.07
3,家用电器,3.46
4,母婴,3.33
5,手机数码,2.52
6,食品饮料,1.36


In [ ]:
get_table_columns("fact_refund")

'refund_id,order_item_id,order_id,user_id,apply_date_id,apply_time,refund_type,refund_reason,refund_status,refund_quantity,refund_amount,finish_time'

In [ ]:
q("""
select c.category_l1,round(sum(coalesce(r.refund_amount,0))/sum(i.pay_amount) * 100 ,2) as `退款率`
from fact_order_item i
join fact_order o on o.order_id = i.order_id and o.pay_time is not null
join dim_sku s on i.sku_id = s.sku_id
join dim_category c on s.category_id = c.category_id
left join fact_refund r on i.order_item_id = r.order_item_id and r.refund_status = "退款成功"
group by c.category_l1
order by `退款率`
""")

,category_l1,退款率
0,食品饮料,1.36
1,手机数码,2.52
2,母婴,3.33
3,家用电器,3.46
4,家居日用,4.07
5,美妆个护,4.42
6,服饰鞋包,11.54


## 7. 外键完整性自检（所有孤儿行数应为 0）

In [ ]:
q("""
SELECT 'order.user_id' AS k, COUNT(*) AS bad
FROM fact_order o LEFT JOIN dim_user d USING (user_id) WHERE d.user_id IS NULL
UNION ALL
SELECT 'item.sku_id', COUNT(*)
FROM fact_order_item i LEFT JOIN dim_sku s USING (sku_id) WHERE s.sku_id IS NULL
UNION ALL
SELECT 'refund.order_item_id', COUNT(*)
FROM fact_refund r LEFT JOIN fact_order_item i USING (order_item_id) WHERE i.order_item_id IS NULL
UNION ALL
SELECT 'behavior.user_id', COUNT(*)
FROM fact_user_behavior b LEFT JOIN dim_user d USING (user_id) WHERE d.user_id IS NULL
""")

,k,bad
0,order.user_id,0
1,item.sku_id,0
2,refund.order_item_id,0
3,behavior.user_id,0


## 8. 自由查询

下面这条只是模板，改 SQL 直接重跑。

In [ ]:
q("""
SELECT platform,
       COUNT(DISTINCT order_id) AS orders,
       ROUND(SUM(order_amount), 2) AS gmv
FROM fact_order
WHERE pay_time IS NOT NULL
GROUP BY platform
ORDER BY gmv DESC
""")

,platform,orders,gmv
0,APP,32437,49715166.18
1,小程序,12343,19086138.65
2,PC,5571,8499080.35
3,H5,4145,6271148.86


In [8]:
get_table_columns("fact_order_item")

'order_item_id,order_id,sku_id,activity_id,quantity,unit_price,item_amount,discount_amount,pay_amount'

In [16]:
q("""
with cat as (
select s.sku_id,c.category_l1 as category_l1
from 
dim_sku s 
join dim_category c on s.category_id = c.category_id
group by s.sku_id,c.category_l1
),

sku_sales as (
select t.category_l1 as category_l1,i.sku_id as sku_id,sum(i.pay_amount) as sales
from fact_order_item i 
join cat t on i.sku_id = t.sku_id
join fact_order o on i.order_id = o.order_id 
where o.pay_time is not null and date(o.pay_time) >= '2025-01-01' and date(o.pay_time) <= '2025-12-31'
group by t.category_l1,i.sku_id),

acc as (select 
category_l1,sku_id,sales,sum(sales) over(partition by category_l1 order by sales desc) as acc_sales,
sum(sales) over(partition by category_l1) as cat_sales
from 
sku_sales),

rk as (select category_l1,
cat_sales,
sales,
acc_sales,
sku_id,
case when acc_sales - sales < 0.8 * cat_sales then 1 else 0 end as is_core
from 
acc) 

select category_l1,
cat_sales,
count(sku_id) as sku_cnt,
sum(is_core) as core_sku_cnt,
round(sum(case when is_core = 1 then sales else 0 end)/cat_sales*100,2) as core_share_pct
from rk 
group by category_l1,cat_sales
order by cat_sales desc
""")

,category_l1,cat_sales,sku_cnt,core_sku_cnt,core_share_pct
0,手机数码,17505016.31,44,16.0,81.54
1,家用电器,10865482.65,42,18.0,80.14
2,服饰鞋包,9239559.60,52,24.0,80.81
3,美妆个护,4699473.41,37,14.0,81.09
4,家居日用,1645673.67,41,17.0,81.33
5,母婴,1505125.67,52,31.0,81.19
6,食品饮料,964527.47,46,26.0,80.02


In [ ]:
q("""
select s.sku_id,c.category_l1 as category_l1
from 
dim_sku s 
join dim_category c on s.category_id = c.category_id
group by s.sku_id,c.category_l1
""")

,sku_id,category_l1
0,K00008,手机数码
1,K00009,手机数码
2,K00010,手机数码
3,K00025,手机数码
4,K00026,手机数码
...,...,...
312,K00298,家居日用
313,K00306,家居日用
314,K00314,家居日用
315,K00315,家居日用
